In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"

import json
import math
import time
import random
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
from torchvision.datasets import ImageFolder

from sklearn.metrics import accuracy_score, classification_report

import optuna
import timm
from timm.data import resolve_model_data_config, create_transform


# =========================================================
# 0) Config
# =========================================================
@dataclass
class CFG:
    # ---------- paths ----------
    BASE_DIR: str = "/kaggle/input/datasets/carson650/wildfire/wildfire"
    TRAIN_SPLIT: str = "train/images"
    VAL_SPLIT: str = "valid/images"
    TEST_SPLIT: str = "test"
    OUT_DIR: str = "./runs_flamevision_resmlp"

    # ---------- model ----------
    MODEL_NAME: str = "resmlp_12_224.fb_in1k"
    NUM_CLASSES: int = 2

    # 若你已经手动下载好了本地权重文件, 在这里填绝对路径
    # 例如: "/root/autodl-tmp/resmlp_12_no_dist.pth"
    # 留空则走 direct URL
    LOCAL_PRETRAINED_PATH: str = ""

    # 是否使用预训练
    USE_PRETRAINED: bool = True

    # ---------- reproducibility ----------
    SEED: int = 42

    # ---------- hardware ----------
    NUM_WORKERS: int = 4
    USE_AMP: bool = True

    # ---------- optuna ----------
    N_TRIALS: int = 10
    MAX_EPOCHS_TUNE: int = 10
    EARLY_STOP_PATIENCE: int = 3
    OPTUNA_TIMEOUT_SEC: int = 0
    STUDY_NAME: str = "flamevision_resmlp_ce"

    # ---------- logging ----------
    PRINT_EVERY: int = 1

In [2]:
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def resolve_split_paths(cfg: CFG) -> Tuple[str, str, str]:
    train_dir = os.path.join(cfg.BASE_DIR, cfg.TRAIN_SPLIT)
    val_dir = os.path.join(cfg.BASE_DIR, cfg.VAL_SPLIT)
    test_dir = os.path.join(cfg.BASE_DIR, cfg.TEST_SPLIT)

    for p in [train_dir, val_dir, test_dir]:
        if not os.path.isdir(p):
            raise FileNotFoundError(f"Split directory not found: {p}")

    return train_dir, val_dir, test_dir


def sanity_check_imagefolder(ds: ImageFolder, split_name: str):
    if len(ds.classes) < 2:
        raise RuntimeError(
            f"[{split_name}] only found {len(ds.classes)} classes. "
            f"ImageFolder 需要 split/class_name/*.png 这样的结构。"
        )
    if len(ds) == 0:
        raise RuntimeError(f"[{split_name}] dataset is empty.")


def make_loader(ds, batch_size: int, shuffle: bool, num_workers: int, device: torch.device) -> DataLoader:
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )


def compute_class_weights_from_targets(targets: List[int], num_classes: int) -> torch.Tensor:
    counts = np.bincount(np.array(targets), minlength=num_classes).astype(np.float64)
    if (counts == 0).any():
        raise RuntimeError(f"Some class has zero samples: counts={counts.tolist()}")

    N = counts.sum()
    weights = N / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

In [3]:
def get_pretrained_cfg_overlay(cfg: CFG) -> Dict[str, Any]:
    """
    关键:
    - 显式关闭 hf_hub_id
    - 如果给了本地文件, 优先走本地文件
    - 如果没给本地文件, 则回退到 direct URL
    """
    overlay = {"hf_hub_id": None}

    if cfg.LOCAL_PRETRAINED_PATH and os.path.isfile(cfg.LOCAL_PRETRAINED_PATH):
        overlay["file"] = cfg.LOCAL_PRETRAINED_PATH

    return overlay


def get_resmlp_model_and_transforms(cfg: CFG, device: torch.device):
    overlay = get_pretrained_cfg_overlay(cfg)

    model = timm.create_model(
        cfg.MODEL_NAME,
        pretrained=cfg.USE_PRETRAINED,
        num_classes=cfg.NUM_CLASSES,
        pretrained_cfg_overlay=overlay
    ).to(device)

    data_config = resolve_model_data_config(model)
    train_tfms = create_transform(**data_config, is_training=True)
    eval_tfms = create_transform(**data_config, is_training=False)

    return model, train_tfms, eval_tfms, data_config, overlay


def get_resmlp_model_only(cfg: CFG, device: torch.device):
    overlay = get_pretrained_cfg_overlay(cfg)

    model = timm.create_model(
        cfg.MODEL_NAME,
        pretrained=cfg.USE_PRETRAINED,
        num_classes=cfg.NUM_CLASSES,
        pretrained_cfg_overlay=overlay
    ).to(device)

    return model



def format_metrics(report: Dict[str, Any], class_names: List[str]) -> Dict[str, Any]:
    overall = {
        "accuracy": float(report["accuracy"]),
        "recall": float(report["macro avg"]["recall"]),
        "f1": float(report["macro avg"]["f1-score"]),
    }

    per_class = {}
    for cname in class_names:
        per_class[cname] = {
            "precision": float(report[cname]["precision"]),
            "recall": float(report[cname]["recall"]),
            "f1": float(report[cname]["f1-score"]),
            "support": int(report[cname]["support"]),
        }

    return {"overall": overall, "per_class": per_class}


@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_true = []
    all_pred = []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        logits = model(xb)
        pred = torch.argmax(logits, dim=1).cpu().numpy()

        all_true.append(yb.numpy())
        all_pred.append(pred)

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    return y_true, y_pred


def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, class_names: List[str]) -> Dict[str, Any]:
    y_true, y_pred = predict(model, loader, device)

    rep = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    rep["accuracy"] = float(accuracy_score(y_true, y_pred))

    return {
        "report_dict": rep,
        "summary": format_metrics(rep, class_names),
        "y_true": y_true,
        "y_pred": y_pred,
    }



def build_warmup_cosine_scheduler(optimizer, total_epochs: int, warmup_epochs: int):
    def lr_lambda(current_epoch: int):
        if warmup_epochs > 0 and current_epoch < warmup_epochs:
            return float(current_epoch + 1) / float(max(1, warmup_epochs))

        if total_epochs == warmup_epochs:
            return 1.0

        progress = (current_epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        progress = min(max(progress, 0.0), 1.0)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

In [4]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    use_amp: bool = True,
) -> float:
    model.train()
    scaler = torch.amp.GradScaler("cuda")

    running_loss = 0.0
    total_n = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(xb)
            loss = criterion(logits, yb)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        bs = xb.size(0)
        running_loss += float(loss.item()) * bs
        total_n += bs

    return running_loss / max(1, total_n)

In [5]:
def run_optuna_tuning(cfg: CFG) -> Dict[str, Any]:
    ensure_dir(cfg.OUT_DIR)
    set_seed(cfg.SEED)
    device = get_device()

    tmp_model, train_tfms, eval_tfms, data_config, overlay = get_resmlp_model_and_transforms(cfg, device)
    del tmp_model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    train_dir, val_dir, _ = resolve_split_paths(cfg)

    ds_train = ImageFolder(train_dir, transform=train_tfms)
    ds_val = ImageFolder(val_dir, transform=eval_tfms)

    sanity_check_imagefolder(ds_train, "train")
    sanity_check_imagefolder(ds_val, "valid")

    if ds_train.class_to_idx != ds_val.class_to_idx:
        raise RuntimeError(
            f"class_to_idx mismatch!\n"
            f"train: {ds_train.class_to_idx}\n"
            f"valid: {ds_val.class_to_idx}"
        )

    class_names = ds_train.classes
    class_weights = compute_class_weights_from_targets(ds_train.targets, num_classes=len(class_names)).to(device)

    print("===== Dataset Info =====")
    print("BASE_DIR:", cfg.BASE_DIR)
    print("MODEL_NAME:", cfg.MODEL_NAME)
    print("LOCAL_PRETRAINED_PATH:", cfg.LOCAL_PRETRAINED_PATH if cfg.LOCAL_PRETRAINED_PATH else "[None -> direct URL]")
    print("USE_PRETRAINED:", cfg.USE_PRETRAINED)
    print("pretrained_cfg_overlay:", overlay)
    print("train samples:", len(ds_train))
    print("valid samples:", len(ds_val))
    print("classes:", class_names)
    print("class_to_idx:", ds_train.class_to_idx)
    print("class_weights:", class_weights.detach().cpu().numpy().round(6).tolist())
    print("data_config:", data_config)
    print("device:", device)

    def objective(trial: optuna.Trial) -> float:
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
        lr = trial.suggest_float("lr", 5e-5, 1e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-2, 2e-1, log=True)
        warmup_epochs = trial.suggest_int("warmup_epochs", 0, 3)

        max_epochs = cfg.MAX_EPOCHS_TUNE
        patience = cfg.EARLY_STOP_PATIENCE

        dl_train = make_loader(ds_train, batch_size=batch_size, shuffle=True,
                               num_workers=cfg.NUM_WORKERS, device=device)
        dl_val = make_loader(ds_val, batch_size=batch_size, shuffle=False,
                             num_workers=cfg.NUM_WORKERS, device=device)

        model = get_resmlp_model_only(cfg, device)

        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = build_warmup_cosine_scheduler(
            optimizer=optimizer,
            total_epochs=max_epochs,
            warmup_epochs=warmup_epochs
        )

        best_val_f1 = -1.0
        best_epoch = 0
        bad_epochs = 0

        for epoch in range(1, max_epochs + 1):
            t0 = time.time()

            train_loss = train_one_epoch(
                model=model,
                loader=dl_train,
                criterion=criterion,
                optimizer=optimizer,
                device=device,
                use_amp=cfg.USE_AMP
            )

            eval_out = evaluate(model, dl_val, device, class_names)
            val_summary = eval_out["summary"]
            val_f1 = val_summary["overall"]["f1"]
            val_acc = val_summary["overall"]["accuracy"]
            val_recall = val_summary["overall"]["recall"]

            scheduler.step()

            trial.report(val_f1, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            improved = val_f1 > best_val_f1 + 1e-6
            if improved:
                best_val_f1 = val_f1
                best_epoch = epoch
                bad_epochs = 0
            else:
                bad_epochs += 1

            if epoch % cfg.PRINT_EVERY == 0:
                lr_now = optimizer.param_groups[0]["lr"]
                print(
                    f"[trial {trial.number:03d}] "
                    f"epoch {epoch:02d}/{max_epochs} "
                    f"loss={train_loss:.4f} "
                    f"val_acc={val_acc:.4f} "
                    f"val_recall(macro)={val_recall:.4f} "
                    f"val_f1(macro)={val_f1:.4f} "
                    f"lr={lr_now:.2e} "
                    f"time={time.time()-t0:.1f}s"
                )

            if bad_epochs >= patience:
                break

        trial.set_user_attr("best_epoch", int(best_epoch))
        return float(best_val_f1)

    sampler = optuna.samplers.TPESampler(seed=cfg.SEED)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name=cfg.STUDY_NAME
    )

    print("\n===== Optuna Tuning Start =====")
    study.optimize(
        objective,
        n_trials=cfg.N_TRIALS,
        timeout=(cfg.OPTUNA_TIMEOUT_SEC if cfg.OPTUNA_TIMEOUT_SEC > 0 else None),
        show_progress_bar=True
    )

    best_trial = study.best_trial
    best_params = dict(best_trial.params)
    best_epoch = int(best_trial.user_attrs.get("best_epoch", cfg.MAX_EPOCHS_TUNE))

    print("\n===== Best Trial =====")
    print("best_value (val macro-F1):", float(best_trial.value))
    print("best_epoch:", best_epoch)
    print("best_params:", json.dumps(best_params, indent=2))

    trials_df = study.trials_dataframe()
    trials_df.to_csv(os.path.join(cfg.OUT_DIR, "optuna_trials_resmlp.csv"), index=False)

    best_json = {
        "best_value_val_macro_f1": float(best_trial.value),
        "best_epoch": int(best_epoch),
        "best_params": best_params,
        "class_names": class_names,
        "class_to_idx": ds_train.class_to_idx,
        "model_name": cfg.MODEL_NAME,
        "data_config": data_config,
        "pretrained_cfg_overlay": overlay,
    }
    with open(os.path.join(cfg.OUT_DIR, "optuna_best_resmlp.json"), "w", encoding="utf-8") as f:
        json.dump(best_json, f, indent=2)

    return {
        "best_params": best_params,
        "best_epoch": best_epoch,
        "class_names": class_names,
        "class_to_idx": ds_train.class_to_idx,
        "data_config": data_config,
        "pretrained_cfg_overlay": overlay,
    }

In [6]:
def final_train_and_test(cfg: CFG, best: Dict[str, Any]):
    ensure_dir(cfg.OUT_DIR)
    set_seed(cfg.SEED)
    device = get_device()

    tmp_model, train_tfms, eval_tfms, data_config, overlay = get_resmlp_model_and_transforms(cfg, device)
    del tmp_model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    train_dir, val_dir, test_dir = resolve_split_paths(cfg)

    class_names = best["class_names"]
    best_params = best["best_params"]
    best_epoch = int(best["best_epoch"])

    ds_train = ImageFolder(train_dir, transform=train_tfms)
    ds_val_as_train = ImageFolder(val_dir, transform=train_tfms)
    ds_test = ImageFolder(test_dir, transform=eval_tfms)

    if ds_train.class_to_idx != ds_val_as_train.class_to_idx or ds_train.class_to_idx != ds_test.class_to_idx:
        raise RuntimeError(
            "class_to_idx mismatch across splits.\n"
            f"train: {ds_train.class_to_idx}\n"
            f"valid: {ds_val_as_train.class_to_idx}\n"
            f"test : {ds_test.class_to_idx}"
        )

    ds_final = ConcatDataset([ds_train, ds_val_as_train])

    merged_targets = list(ds_train.targets) + list(ds_val_as_train.targets)
    class_weights = compute_class_weights_from_targets(merged_targets, num_classes=len(class_names)).to(device)

    batch_size = int(best_params["batch_size"])
    lr = float(best_params["lr"])
    weight_decay = float(best_params["weight_decay"])
    warmup_epochs = int(best_params["warmup_epochs"])

    dl_final = make_loader(ds_final, batch_size=batch_size, shuffle=True,
                           num_workers=cfg.NUM_WORKERS, device=device)
    dl_test = make_loader(ds_test, batch_size=batch_size, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, device=device)

    model = get_resmlp_model_only(cfg, device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = build_warmup_cosine_scheduler(
        optimizer=optimizer,
        total_epochs=best_epoch,
        warmup_epochs=warmup_epochs
    )

    print("\n===== Final Training on train+valid =====")
    print("MODEL_NAME:", cfg.MODEL_NAME)
    print("LOCAL_PRETRAINED_PATH:", cfg.LOCAL_PRETRAINED_PATH if cfg.LOCAL_PRETRAINED_PATH else "[None -> direct URL]")
    print("USE_PRETRAINED:", cfg.USE_PRETRAINED)
    print("pretrained_cfg_overlay:", overlay)
    print("final_epochs:", best_epoch)
    print("best_params:", json.dumps(best_params, indent=2))
    print("data_config:", data_config)

    for epoch in range(1, best_epoch + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=dl_final,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            use_amp=cfg.USE_AMP
        )
        scheduler.step()

        if epoch % cfg.PRINT_EVERY == 0:
            lr_now = optimizer.param_groups[0]["lr"]
            print(
                f"[final] epoch {epoch:02d}/{best_epoch} "
                f"loss={train_loss:.4f} "
                f"lr={lr_now:.2e}"
            )

    model_path = os.path.join(cfg.OUT_DIR, "final_model_resmlp.pth")
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "class_to_idx": ds_train.class_to_idx,
            "best_params": best_params,
            "final_epochs": best_epoch,
            "cfg": asdict(cfg),
            "data_config": data_config,
            "pretrained_cfg_overlay": overlay,
        },
        model_path
    )
    print("Saved final model:", model_path)

    print("\n===== Final Test Evaluation =====")
    test_out = evaluate(model, dl_test, device, class_names)
    summary = test_out["summary"]
    report_dict = test_out["report_dict"]

    print("\n[Overall]")
    print("accuracy:", round(summary["overall"]["accuracy"], 6))
    print("recall  :", round(summary["overall"]["recall"], 6), " (macro)")
    print("f1      :", round(summary["overall"]["f1"], 6), " (macro)")

    print("\n[Per-class]")
    for cname in class_names:
        m = summary["per_class"][cname]
        print(
            f"{cname:>12} | "
            f"P={m['precision']:.6f} "
            f"R={m['recall']:.6f} "
            f"F1={m['f1']:.6f} "
            f"(support={m['support']})"
        )

    with open(os.path.join(cfg.OUT_DIR, "test_report_resmlp.json"), "w", encoding="utf-8") as f:
        json.dump(
            {
                "summary": summary,
                "classification_report": report_dict,
            },
            f,
            indent=2
        )

    rep_df = pd.DataFrame(report_dict).T
    rep_df.to_csv(os.path.join(cfg.OUT_DIR, "test_report_table_resmlp.csv"), index=True)

    print("\nSaved test reports:")
    print(" -", os.path.join(cfg.OUT_DIR, "test_report_resmlp.json"))
    print(" -", os.path.join(cfg.OUT_DIR, "test_report_table_resmlp.csv"))

In [7]:
def main():
    cfg = CFG()
    ensure_dir(cfg.OUT_DIR)

    train_dir, val_dir, test_dir = resolve_split_paths(cfg)

    print("===== Paths =====")
    print("TRAIN:", train_dir, "->", os.listdir(train_dir))
    print("VALID:", val_dir, "->", os.listdir(val_dir))
    print("TEST :", test_dir, "->", os.listdir(test_dir))

    best = run_optuna_tuning(cfg)
    final_train_and_test(cfg, best)


if __name__ == "__main__":
    main()

===== Paths =====
TRAIN: /kaggle/input/datasets/carson650/wildfire/wildfire/train/images -> ['nofire', 'fire']
VALID: /kaggle/input/datasets/carson650/wildfire/wildfire/valid/images -> ['nofire', 'fire']
TEST : /kaggle/input/datasets/carson650/wildfire/wildfire/test -> ['nofire', 'fire']
Downloading: "https://dl.fbaipublicfiles.com/deit/resmlp_12_no_dist.pth" to /root/.cache/torch/hub/checkpoints/resmlp_12_no_dist.pth


[I 2026-03-12 07:59:40,340] A new study created in memory with name: flamevision_resmlp_ce


===== Dataset Info =====
BASE_DIR: /kaggle/input/datasets/carson650/wildfire/wildfire
MODEL_NAME: resmlp_12_224.fb_in1k
LOCAL_PRETRAINED_PATH: [None -> direct URL]
USE_PRETRAINED: True
pretrained_cfg_overlay: {'hf_hub_id': None}
train samples: 11198
valid samples: 2000
classes: ['fire', 'nofire']
class_to_idx: {'fire': 0, 'nofire': 1}
class_weights: [0.9655110239982605, 1.0370440483093262]
data_config: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.875, 'crop_mode': 'center'}
device: cuda

===== Optuna Tuning Start =====


  0%|          | 0/10 [00:00<?, ?it/s]

[trial 000] epoch 01/10 loss=0.8687 val_acc=0.9460 val_recall(macro)=0.9315 val_f1(macro)=0.9410 lr=2.93e-04 time=71.3s
[trial 000] epoch 02/10 loss=0.2230 val_acc=0.9450 val_recall(macro)=0.9291 val_f1(macro)=0.9397 lr=2.72e-04 time=64.0s
[trial 000] epoch 03/10 loss=0.1713 val_acc=0.9580 val_recall(macro)=0.9536 val_f1(macro)=0.9551 lr=2.39e-04 time=64.0s
[trial 000] epoch 04/10 loss=0.1290 val_acc=0.9635 val_recall(macro)=0.9644 val_f1(macro)=0.9613 lr=1.97e-04 time=63.8s
[trial 000] epoch 05/10 loss=0.1079 val_acc=0.9580 val_recall(macro)=0.9459 val_f1(macro)=0.9543 lr=1.50e-04 time=63.6s
[trial 000] epoch 06/10 loss=0.0912 val_acc=0.9785 val_recall(macro)=0.9769 val_f1(macro)=0.9771 lr=1.04e-04 time=63.7s
[trial 000] epoch 07/10 loss=0.0741 val_acc=0.9845 val_recall(macro)=0.9817 val_f1(macro)=0.9834 lr=6.19e-05 time=63.7s
[trial 000] epoch 08/10 loss=0.0529 val_acc=0.9870 val_recall(macro)=0.9845 val_f1(macro)=0.9861 lr=2.87e-05 time=64.4s
[trial 000] epoch 09/10 loss=0.0431 val_